In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Veri setlerini yükle
orders = pd.read_csv('../data/raw/olist_orders_dataset.csv')
order_items = pd.read_csv('../data/raw/olist_order_items_dataset.csv')
products = pd.read_csv('../data/raw/olist_products_dataset.csv')
customers = pd.read_csv('../data/raw/olist_customers_dataset.csv')
sellers = pd.read_csv('../data/raw/olist_sellers_dataset.csv')
reviews = pd.read_csv('../data/raw/olist_order_reviews_dataset.csv')
payments = pd.read_csv('../data/raw/olist_order_payments_dataset.csv')
category_translation = pd.read_csv('../data/raw/product_category_name_translation.csv')

# Tarih dönüşümleri
date_columns = ['order_purchase_timestamp', 'order_approved_at',
                'order_delivered_carrier_date', 'order_delivered_customer_date',
                'order_estimated_delivery_date']
for col in date_columns:
    orders[col] = pd.to_datetime(orders[col])

order_items['shipping_limit_date'] = pd.to_datetime(order_items['shipping_limit_date'])

# Sadece delivered siparişler
orders_delivered = orders[orders['order_status'] == 'delivered'].copy()

# Tabloları birleştir
df = orders_delivered.merge(order_items, on='order_id', how='inner')
df = df.merge(products, on='product_id', how='left')
df = df.merge(category_translation, on='product_category_name', how='left')
df = df.merge(customers[['customer_id', 'customer_state', 'customer_city']], on='customer_id', how='left')
df = df.merge(sellers[['seller_id', 'seller_state', 'seller_city']], on='seller_id', how='left')

# Tarih kırpma
df = df[(df['order_purchase_timestamp'] >= '2017-01-01') &
        (df['order_purchase_timestamp'] <= '2018-08-31')]

print(f"Veri hazır: {df.shape}")

Veri hazır: (109880, 27)


In [2]:
print("Önce eksik veri durumu:")
print(df.isnull().sum()[df.isnull().sum() > 0])

# 1. Ürün kategorisi eksik olanları "unknown" yap
df['product_category_name'] = df['product_category_name'].fillna('unknown')
df['product_category_name_english'] = df['product_category_name_english'].fillna('unknown')

# 2. Ürün boyut/ağırlık eksiklerini medyan ile doldur
size_cols = ['product_weight_g', 'product_length_cm', 'product_height_cm', 'product_width_cm']
for col in size_cols:
    median_val = df[col].median()
    df[col] = df[col].fillna(median_val)
    print(f"{col} eksikleri {median_val:.1f} ile dolduruldu")

# 3. Diğer ürün özelliklerini medyan ile doldur
other_cols = ['product_name_lenght', 'product_description_lenght', 'product_photos_qty']
for col in other_cols:
    df[col] = df[col].fillna(df[col].median())

print("\nSonra eksik veri durumu:")
print(df.isnull().sum()[df.isnull().sum() > 0])
print("\nEksik veri kalmadı mı?", df.isnull().sum().sum() == 0)

Önce eksik veri durumu:
order_approved_at                  15
order_delivered_carrier_date        2
order_delivered_customer_date       8
product_category_name            1535
product_name_lenght              1535
product_description_lenght       1535
product_photos_qty               1535
product_weight_g                   18
product_length_cm                  18
product_height_cm                  18
product_width_cm                   18
product_category_name_english    1557
dtype: int64
product_weight_g eksikleri 700.0 ile dolduruldu
product_length_cm eksikleri 25.0 ile dolduruldu
product_height_cm eksikleri 13.0 ile dolduruldu
product_width_cm eksikleri 20.0 ile dolduruldu

Sonra eksik veri durumu:
order_approved_at                15
order_delivered_carrier_date      2
order_delivered_customer_date     8
dtype: int64

Eksik veri kalmadı mı? False


In [3]:
# Tarih sütunlarındaki eksikleri düşür
# Bunlar teslimat tarihi olmayan siparişler, modelimiz için güvenilmez
before = len(df)
df = df.dropna(subset=['order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date'])
after = len(df)

print(f"Düşürülen satır sayısı: {before - after}")
print(f"Kalan satır sayısı: {after}")
print(f"\nEksik veri kaldı mı? {df.isnull().sum().sum() > 0}")

Düşürülen satır sayısı: 24
Kalan satır sayısı: 109856

Eksik veri kaldı mı? False


In [5]:
print("AYKIRI DEĞER TEMİZLEME")
print("="*50)

# 1. Fiyat için alt sınır — 1 BRL altı anlamsız
before = len(df)
df = df[df['price'] >= 1.0]
print(f"1 BRL altı fiyatlar düşürüldü: {before - len(df)} satır")

# 2. Fiyat için üst sınır — kategori bazında IQR yöntemi
def remove_price_outliers(group):
    Q1 = group['price'].quantile(0.25)
    Q3 = group['price'].quantile(0.75)
    IQR = Q3 - Q1
    upper = Q3 + 3 * IQR
    return group[group['price'] <= upper]

before = len(df)
df = df.groupby('product_category_name_english', group_keys=False).apply(
    lambda x: remove_price_outliers(x), include_groups=False
)
print(f"Kategori bazlı aykırı fiyatlar düşürüldü: {before - len(df)} satır")

# 3. Kargo fiyatı sıfır olamaz
before = len(df)
df = df[df['freight_value'] >= 0]
print(f"Negatif kargo değerleri düşürüldü: {before - len(df)} satır")

print(f"\nFinal veri boyutu: {df.shape}")
print(f"\nFiyat istatistikleri (temizlenmiş):")
print(df['price'].describe().round(2))

AYKIRI DEĞER TEMİZLEME
1 BRL altı fiyatlar düşürüldü: 0 satır
Kategori bazlı aykırı fiyatlar düşürüldü: 937 satır
Negatif kargo değerleri düşürüldü: 0 satır

Final veri boyutu: (104592, 26)

Fiyat istatistikleri (temizlenmiş):
count    104592.00
mean         94.93
std          93.88
min           1.20
25%          39.00
50%          69.90
75%         120.00
max        3399.99
Name: price, dtype: float64


In [6]:
# Teslimat süresi hesapla ve mantıksız olanları temizle
df['delivery_days'] = (df['order_delivered_customer_date'] - 
                       df['order_purchase_timestamp']).dt.days

print("Teslimat süresi istatistikleri:")
print(df['delivery_days'].describe().round(1))

# Negatif veya sıfır teslimat süresi olamaz
before = len(df)
df = df[df['delivery_days'] > 0]
print(f"\nNegatif teslimat süresi düşürüldü: {before - len(df)} satır")

# 200 günden uzun teslimat da anlamsız
before = len(df)
df = df[df['delivery_days'] <= 200]
print(f"200 gün üzeri teslimat düşürüldü: {before - len(df)} satır")

print(f"\nFinal veri boyutu: {df.shape}")

Teslimat süresi istatistikleri:
count    104592.0
mean         11.9
std           9.4
min           0.0
25%           6.0
50%          10.0
75%          15.0
max         209.0
Name: delivery_days, dtype: float64

Negatif teslimat süresi düşürüldü: 18 satır
200 gün üzeri teslimat düşürüldü: 2 satır

Final veri boyutu: (104572, 27)
